# Issam — Arabic Words Live Test (100 classes, KARSL reference videos)

**Arabic words only** — issamjebnouni pre-trained model (SignID **71–170**, medical/body vocabulary).
Same layout as `ArSL_Word_Class_Guide.ipynb`:

| Left | Right top | Right bottom |
|------|-----------|---------------|
| Your webcam | **KARSL dataset** reference video (loops) | Top probability bars |

### Run order
Cell **1 → 2 → 3 → 4** (optional video sanity) → **5** (live guided test)

### Live keys (Cell 5)
| Key | Action |
|-----|--------|
| `N` / `→` | Next word |
| `P` / `←` | Previous word |
| `K` | Next dataset sample video |
| `V` | Classify **current reference** `.mp4` in console |
| `C` | Clear capture buffer |
| `O` | Mark class **OK** |
| `R` | Mark class **needs retest** |
| `U` | Clear class status (untested) |
| `G` | Toggle brightness |
| `Q` | Quit (saves checklist) |

**Portable bundle** (`Issam_Demo_Bundle/`) — model, labels, and `sample_videos/skeleton_SignID0071.mp4` live here.

Optional full KARSL dataset for reference videos in live test: `E:/Downloads/Arabic Words Dataset` (edit in Cell 2).

Quick video-only test (no webcam): run `Issam_One_Video_Sample.ipynb` in this folder.

In [6]:
# CELL 1 — imports + Arabic overlay
import os
import cv2
import time
import numpy as np
import pandas as pd
import mediapipe as mp
import tensorflow as tf
from pathlib import Path
from collections import deque
from PIL import Image, ImageDraw, ImageFont
import arabic_reshaper
from bidi.algorithm import get_display

os.environ.setdefault('PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION', 'python')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

print(f'TF {tf.__version__}  |  MP {mp.__version__}  |  CV2 {cv2.__version__}')

_FONT_CANDS = [r'C:/Windows/Fonts/segoeui.ttf', r'C:/Windows/Fonts/tahoma.ttf', r'C:/Windows/Fonts/arial.ttf']
_BASE_FONT = next((p for p in _FONT_CANDS if Path(p).exists()), None)
_font_cache = {}

def _has_arabic(t):
    return any('\u0600' <= c <= '\u06FF' for c in str(t))

def _shape_ar(text):
    s = str(text)
    if not _has_arabic(s):
        return s
    try:
        return get_display(arabic_reshaper.reshape(s))
    except Exception:
        return s

def _get_font(size):
    if size not in _font_cache:
        _font_cache[size] = ImageFont.truetype(_BASE_FONT, size) if _BASE_FONT else ImageFont.load_default()
    return _font_cache[size]

def apply_pil_texts(bgr, ops):
    if not ops:
        return bgr
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img)
    for text, pos, size, color in ops:
        draw.text(pos, _shape_ar(text), font=_get_font(size), fill=(color[2], color[1], color[0]))
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def _clean_karsl_label(val):
    s = str(val).strip()
    return '' if not s or s.lower() in ('nan', 'none') else s

def _display_label(ar, en):
    """Primary Arabic; fallback English."""
    ar, en = _clean_karsl_label(ar), _clean_karsl_label(en)
    return ar if ar and _has_arabic(ar) else (en or '?')

def _truncate_en(text, n=22):
    t = _clean_karsl_label(text)
    return t if len(t) <= n else t[: n - 1] + '…'

print('Setup OK')


TF 2.10.0  |  MP 0.10.9  |  CV2 4.11.0
Setup OK


In [7]:
# CELL 2 — model + 100 Arabic word labels (self-contained bundle paths)
import sys
from bundle_config import find_bundle_from_cwd, MODEL_PATH, LABELS_XLSX, BUNDLE_SAMPLES
from bundle_config import SAMPLE_VIDEO, MANIFEST_CSV, CHECKLIST_PATH, DATASET_ROOT, F_AVG, N_FEATURES

DIR = find_bundle_from_cwd()
print(f'Bundle root: {DIR}')
SKELETON_MP4 = SAMPLE_VIDEO if SAMPLE_VIDEO.exists() else DIR / 'skeleton.mp4'
START_CLASS_INDEX = 0
CAMERA_INDEX = 0
CAMERA_WIDTH, CAMERA_HEIGHT = 640, 480
CONFIDENCE_THRESHOLD = 0.35
DISPLAY_MIN_CONF = 0.15
PREDICTION_INTERVAL = 0.32
USE_DETECT_BRIGHTNESS = True
BRIGHTNESS_ALPHA, BRIGHTNESS_BETA = 1.25, 30

karsl_df = pd.read_excel(LABELS_XLSX)
karsl_100 = karsl_df.iloc[70:170].reset_index(drop=True)
words_en = np.array([_clean_karsl_label(v) for v in karsl_100['Sign-English']])
words_ar = np.array([_clean_karsl_label(v) for v in karsl_100['Sign-Arabic']])
sign_ids = karsl_100['SignID'].astype(int).to_numpy()
num_classes = len(words_ar)
class_list = [(i, words_en[i], words_ar[i], int(sign_ids[i])) for i in range(num_classes)]

# model index i must match KARSL SignID 71..170 in sheet order (issam training layout)
for i in range(num_classes):
    expected_sid = 71 + i
    if sign_ids[i] != expected_sid:
        print(f'WARNING: index {i} SignID {sign_ids[i]} != expected {expected_sid}')

def build_model(n_classes):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(F_AVG, N_FEATURES)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(n_classes, activation='softmax'),
    ])

try:
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    print('[model] loaded full')
except Exception as e:
    print(f'[model] rebuild + weights ({e})')
    model = build_model(num_classes)
    model.load_weights(MODEL_PATH)

if not DATASET_ROOT.exists():
    n_local = sum(1 for sid in sign_ids if (BUNDLE_SAMPLES / f'SignID{int(sid):04d}.mp4').is_file())
    print(f'Using bundled samples: {n_local}/{num_classes} in {BUNDLE_SAMPLES.name}/')
else:
    print(f'KARSL dataset available: {DATASET_ROOT}')

print(f'Model   : {MODEL_PATH.name}')
print(f'Classes : {num_classes} Arabic words (SignID 71–170)')
print(f'Example : [{START_CLASS_INDEX}] {words_ar[START_CLASS_INDEX]} / {words_en[START_CLASS_INDEX]}  (SignID {sign_ids[START_CLASS_INDEX]})')


[model] loaded full
Model   : Holistic_keypoints_BiLSTM_model_3_signers___Date_Time_2023_05_30__12_48_47___Loss_0.026179302483797073___Accuracy_0.9962499737739563.h5
Classes : 100 Arabic words (SignID 71–170)
Example : [0] هيكل عظمي / Skeleton  (SignID 71)


In [8]:
# CELL 3 — issam 225-D extraction + dataset video lookup
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

holistic = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

def frame_for_detection(frame):
    if not globals().get('USE_DETECT_BRIGHTNESS', False):
        return frame
    return cv2.convertScaleAbs(frame, alpha=BRIGHTNESS_ALPHA, beta=BRIGHTNESS_BETA)

def mediapipe_detection(bgr):
    rgb = cv2.cvtColor(frame_for_detection(bgr), cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    return holistic.process(rgb)

def adjust_landmarks(arr, center):
    arr_reshaped = arr.reshape(-1, 3)
    return (arr_reshaped - np.tile(center, (len(arr_reshaped), 1))).reshape(-1)

def extract_keypoints(results):
    pose = np.array([[r.x, r.y, r.z] for r in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(99)
    lh = np.array([[r.x, r.y, r.z] for r in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(63)
    rh = np.array([[r.x, r.y, r.z] for r in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(63)
    return np.concatenate([
        adjust_landmarks(pose, pose[:3]),
        adjust_landmarks(lh, lh[:3]),
        adjust_landmarks(rh, rh[:3]),
    ]).astype(np.float32)

def pad_or_trim(seq, f_avg=F_AVG):
    seq = np.asarray(seq, dtype=np.float32)
    n = min(seq.shape[0], f_avg)
    seq = seq[:n, :]
    while seq.shape[0] < f_avg:
        seq = np.concatenate([seq, seq[-1:, :]], axis=0)
    return seq

def draw_landmarks_on_frame(frame, results):
    out = frame.copy()
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(out, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    if results.left_hand_landmarks:
        mp_drawing.draw_landmarks(out, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    if results.right_hand_landmarks:
        mp_drawing.draw_landmarks(out, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    return out

KARSL_RANGES = ['0001-0070', '0071-0170', '0171-0190', '0191-0300', '0301-0502']

def find_sign_videos(sign_id):
    """Bundled sample first, then full KARSL dataset if available."""
    sid = int(sign_id)
    local = BUNDLE_SAMPLES / f'SignID{sid:04d}.mp4'
    if local.is_file():
        return [local]
    if not DATASET_ROOT.exists():
        return []
    folder_name = f'{sid:04d}'
    found = []
    for split in ('train', 'test'):
        for rng in KARSL_RANGES:
            for base in (DATASET_ROOT / split / rng / folder_name,
                         DATASET_ROOT / split / rng / rng / folder_name):
                if base.is_dir():
                    found.extend(sorted(base.glob('*.mp4')))
                    found.extend(sorted(base.glob('*.avi')))
    return list(dict.fromkeys(found))

test_n = len(find_sign_videos(sign_ids[0]))
print(f'Holistic ready | lookup SignID {sign_ids[0]:04d}: {test_n} dataset videos')


Holistic ready | lookup SignID 0071: 42 dataset videos


In [9]:
# CELL 4 — optional: classify one dataset (or skeleton.mp4) before live

def predict_sign_video(video_path, expected_index=None, topk=5):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(video_path)
    seq = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        seq.append(extract_keypoints(mediapipe_detection(frame)))
    cap.release()
    probs = model.predict(pad_or_trim(seq)[None, ...], verbose=0)[0]
    order = np.argsort(probs)[::-1][:topk]
    print(f'\n=== {Path(video_path).name} ===')
    for rank, i in enumerate(order, 1):
        mark = '  <-- expected' if expected_index is not None and i == expected_index else ''
        print(f'  {rank}. [{i:2d}] SignID {sign_ids[i]}  {words_ar[i]}  /  {words_en[i]}  ({probs[i]*100:.1f}%){mark}')
    if expected_index is not None:
        print(f'  Result: {"CORRECT" if int(order[0]) == expected_index else "WRONG"}')
    return int(order[0]), float(probs[order[0]])

RUN_VIDEO_SANITY = True
SANITY_CLASS_INDEX = START_CLASS_INDEX

if RUN_VIDEO_SANITY:
    vids = find_sign_videos(sign_ids[SANITY_CLASS_INDEX])
    if vids:
        print(f'Class {SANITY_CLASS_INDEX+1}: {words_en[SANITY_CLASS_INDEX]} / {words_ar[SANITY_CLASS_INDEX]}')
        predict_sign_video(vids[0], expected_index=SANITY_CLASS_INDEX)
    elif SKELETON_MP4.exists():
        predict_sign_video(SKELETON_MP4, expected_index=0)
    else:
        print('No dataset video — set DATASET_ROOT or use skeleton.mp4')
else:
    print('Skipped (RUN_VIDEO_SANITY=False)')


Class 1: Skeleton / هيكل عظمي

=== 03_01_0071_(01_12_16_15_52_41)_c.mp4 ===
  1. [ 0] SignID 71  هيكل عظمي  /  Skeleton  (37.9%)  <-- expected
  2. [29] SignID 100  صيدلية  /  pharmacy  (24.0%)
  3. [58] SignID 129  ضغط الدم  /  blood pressure  (13.5%)
  4. [51] SignID 122  أزمة قلبية  /  heart attack  (12.0%)
  5. [21] SignID 92  مستشفى  /  hospital  (4.4%)
  Result: CORRECT


In [10]:
# CELL 5 — GUIDED LIVE TEST (webcam + KARSL reference video + bars)

import json

FONT = cv2.FONT_HERSHEY_SIMPLEX
L_W, L_H = 640, 720
R_W, R_H = 640, 720
REF_HDR_H, TGT_H_R, REF_VID_H, BAR_HDR_H = 36, 46, 288, 22
BAR_Y0 = REF_HDR_H + TGT_H_R + REF_VID_H + BAR_HDR_H
BAR_ROW_H = 36
N_BARS = (R_H - BAR_Y0) // BAR_ROW_H
R_LMARGIN, R_RANK_W, R_LABEL_W, R_BAR_GAP = 12, 28, 168, 8
R_LABEL_X = R_LMARGIN + R_RANK_W + 6
R_BAR_X = R_LABEL_X + R_LABEL_W + R_BAR_GAP
R_PCT_W = 52
R_BAR_W = R_W - R_BAR_X - R_PCT_W - 12
TOP_H_L, BOT_H_L = 96, 88
VID_H_L = L_H - TOP_H_L - BOT_H_L

STATUS_UNSET, STATUS_OK, STATUS_RETEST = 0, 1, 2
STATUS_META = {
    STATUS_UNSET: ('—', (120, 120, 120)),
    STATUS_OK: ('OK', (0, 200, 0)),
    STATUS_RETEST: ('RETEST', (0, 140, 255)),
}


def load_checklist():
    data = {str(i): STATUS_UNSET for i in range(num_classes)}
    if CHECKLIST_PATH.exists():
        try:
            saved = json.loads(CHECKLIST_PATH.read_text(encoding='utf-8'))
            for k, v in saved.get('status', {}).items():
                if k.isdigit() and int(k) < num_classes:
                    data[k] = int(v) if int(v) in STATUS_META else STATUS_UNSET
        except Exception as e:
            print(f'Checklist load warning: {e}')
    return [int(data[str(i)]) for i in range(num_classes)]


def save_checklist(statuses):
    ok_n = sum(1 for s in statuses if s == STATUS_OK)
    retest_n = sum(1 for s in statuses if s == STATUS_RETEST)
    payload = {
        'model': 'issam_100_words',
        'sign_ids': sign_ids.tolist(),
        'status': {str(i): int(statuses[i]) for i in range(num_classes)},
        'summary': {'ok': ok_n, 'retest': retest_n, 'unset': num_classes - ok_n - retest_n},
        'updated': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    CHECKLIST_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return payload['summary']


def checklist_summary(statuses):
    ok_n = sum(1 for s in statuses if s == STATUS_OK)
    retest_n = sum(1 for s in statuses if s == STATUS_RETEST)
    unset_n = num_classes - ok_n - retest_n
    return ok_n, retest_n, unset_n

class RefVideoPlayer:
  NO_VIDEO_FRAME = None
  def __init__(self):
    self.cap = None; self.videos = []; self.vid_idx = 0; self._build_no_video()
  def _build_no_video(self):
    f = np.zeros((REF_VID_H, R_W, 3), dtype=np.uint8)
    cv2.putText(f, 'No reference video', (160, REF_VID_H//2), FONT, 0.75, (100,100,100), 2)
    RefVideoPlayer.NO_VIDEO_FRAME = f
  @property
  def info_str(self):
    if not self.videos: return 'no video'
    return f'{self.vid_idx+1}/{len(self.videos)} {self.videos[self.vid_idx].name}'
  def load(self, sign_id):
    if self.cap: self.cap.release(); self.cap = None
    self.videos = find_sign_videos(sign_id)
    self.vid_idx = 0
    if self.videos: self._open(0)
    return len(self.videos)
  def _open(self, idx):
    if self.cap: self.cap.release()
    self.cap = cv2.VideoCapture(str(self.videos[idx])); self.vid_idx = idx
  def next_sample(self):
    if len(self.videos) > 1: self._open((self.vid_idx + 1) % len(self.videos))
  def get_frame(self):
    if not self.cap or not self.cap.isOpened():
      return RefVideoPlayer.NO_VIDEO_FRAME.copy()
    ret, frame = self.cap.read()
    if not ret:
      self.cap.set(cv2.CAP_PROP_POS_FRAMES, 0); ret, frame = self.cap.read()
    if not ret: return RefVideoPlayer.NO_VIDEO_FRAME.copy()
    return cv2.resize(frame, (R_W, REF_VID_H))
  def close(self):
    if self.cap: self.cap.release(); self.cap = None

def run_guided_issam():
  cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
  cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
  cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)
  cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
  if not cap.isOpened():
    print('Cannot open camera'); return

  ref = RefVideoPlayer()
  buf = deque(maxlen=F_AVG)
  all_probs = []
  class_status = load_checklist()
  current_en = current_ar = ''; current_conf = 0.0
  last_pred = 0.0; hand_ok = False; detect_bright = USE_DETECT_BRIGHTNESS
  fps_hist = deque(maxlen=30)
  target_ci = 0

  GREEN=(0,200,0); RED=(0,0,220); WHITE=(255,255,255); BLACK=(0,0,0)
  GOLD=(0,200,220); LGRAY=(160,160,160); GRAY=(70,70,70); YELLOW=(0,200,255)

  def set_status(ci, st):
    class_status[ci] = st
    lbl, _ = STATUS_META[st]
    print(f'Class {ci+1} marked: {lbl}')

  def load_class(ci):
    nonlocal target_ci, all_probs, current_en, current_ar, current_conf
    target_ci = ci % num_classes
    _, ten, tar, sid = class_list[target_ci]
    n = ref.load(sid)
    buf.clear()
    all_probs = []
    current_en = current_ar = ''
    current_conf = 0.0
    st_lbl, _ = STATUS_META[class_status[target_ci]]
    print(f'Class {target_ci+1}/{num_classes}: {tar} / {ten}  SignID {sid}  {n} videos  [{st_lbl}]')
    return ten, tar, sid

  tgt_en, tgt_ar, tgt_sid = load_class(START_CLASS_INDEX)
  ok_n, retest_n, unset_n = checklist_summary(class_status)
  print(f'Checklist: OK={ok_n}  RETEST={retest_n}  untested={unset_n}  (saved on quit)')
  print('N/P class  K video  V test  O=OK  R=retest  U=clear  C=buffer  G=bright  Q=quit')

  while True:
    t0 = time.time()
    ret, raw = cap.read()
    if not ret: break
    raw = cv2.flip(raw, 1)
    now = time.time()
    results = mediapipe_detection(raw)
    buf.append(extract_keypoints(results))
    hand_ok = bool(results.left_hand_landmarks or results.right_hand_landmarks)

    if len(buf) == F_AVG and hand_ok and (now - last_pred) >= PREDICTION_INTERVAL:
      last_pred = now
      probs = model.predict(np.array(buf, dtype=np.float32)[None, ...], verbose=0)[0]
      order = np.argsort(probs)[::-1]
      all_probs = [(int(i), words_en[i], words_ar[i], float(probs[i])) for i in order]
      top = int(order[0])
      current_en, current_ar, current_conf = words_en[top], words_ar[top], float(probs[top])

    is_match = (current_en == tgt_en and current_conf >= CONFIDENCE_THRESHOLD)
    st = class_status[target_ci]
    st_lbl, st_clr = STATUS_META[st]
    ok_n, retest_n, unset_n = checklist_summary(class_status)

    # --- left panel ---
    left = np.zeros((L_H, L_W, 3), dtype=np.uint8)
    vid = cv2.resize(draw_landmarks_on_frame(raw, results), (L_W, VID_H_L))
    left[TOP_H_L:TOP_H_L + VID_H_L, :] = vid
    fps_hist.append(1.0 / max(time.time() - t0, 1e-6))
    cv2.rectangle(left, (0, 0), (L_W, TOP_H_L), BLACK, -1)
    cv2.putText(left, f'ISSAM  {sum(fps_hist)/len(fps_hist):.0f} FPS', (10, 20), FONT, 0.48, LGRAY, 1)
    cv2.putText(left, f'Class {target_ci+1}/{num_classes}  ID {tgt_sid}', (10, 40), FONT, 0.45, LGRAY, 1)
    cv2.putText(left, f'Hands {"OK" if hand_ok else "NO"}  Buf {len(buf)}/{F_AVG}  Brg {"ON" if detect_bright else "OFF"}',
                (10, 58), FONT, 0.42, LGRAY, 1)
    cv2.putText(left, f'Checklist OK:{ok_n}  Retest:{retest_n}  -:{unset_n}', (10, 76), FONT, 0.40, LGRAY, 1)
    cv2.putText(left, f'Status [{st_lbl}]  O=OK R=retest U=clear', (10, 92), FONT, 0.38, st_clr, 1)
    border = GREEN if is_match else (GOLD if st == STATUS_RETEST else (GRAY if hand_ok else RED))
    cv2.rectangle(left, (0, TOP_H_L), (L_W - 1, TOP_H_L + VID_H_L), border, 3)
    pil_left, pil_right = [], []
    cv2.rectangle(left, (0, L_H - BOT_H_L), (L_W, L_H), BLACK, -1)
    conf_clr = GREEN if current_conf >= CONFIDENCE_THRESHOLD else (0, 140, 255)
    if current_conf >= DISPLAY_MIN_CONF:
      pil_left.append((current_ar, (14, L_H - BOT_H_L + 2), 18, conf_clr))
      cv2.putText(left, _truncate_en(current_en, 26), (14, L_H - BOT_H_L + 44), FONT, 0.44, conf_clr, 1)
      cv2.putText(left, f'{current_conf:.0%}', (14, L_H - 26), FONT, 0.50, conf_clr, 2)
    elif not hand_ok:
      cv2.putText(left, 'No hand — press G', (14, L_H - BOT_H_L + 30), FONT, 0.50, RED, 2)
    cv2.putText(left, 'N/P class  K vid  V test  C buf  G brt  Q quit', (10, L_H - 8), FONT, 0.34, (100, 100, 100), 1)

    # --- right panel ---
    right = np.zeros((R_H, R_W, 3), dtype=np.uint8)
    cv2.rectangle(right, (0, 0), (R_W, REF_HDR_H), (30, 30, 30), -1)
    ref_line = ref.info_str if len(ref.info_str) < 44 else ref.info_str[:41] + '...'
    cv2.putText(right, f'Ref {ref_line}', (R_LMARGIN, 22), FONT, 0.40, LGRAY, 1)
    cv2.putText(right, f'[{st_lbl}]', (R_W - 90, 22), FONT, 0.48, st_clr, 2)
    cv2.rectangle(right, (0, REF_HDR_H), (R_W, REF_HDR_H + TGT_H_R), (20, 20, 20), -1)
    pil_right.append((tgt_ar, (R_LMARGIN, REF_HDR_H + 2), 18, GOLD))
    cv2.putText(right, _truncate_en(tgt_en, 30), (R_LMARGIN, REF_HDR_H + 38), FONT, 0.42, LGRAY, 1)
    right[REF_HDR_H + TGT_H_R:REF_HDR_H + TGT_H_R + REF_VID_H, :] = ref.get_frame()
    cv2.line(right, (0, BAR_Y0 - 2), (R_W, BAR_Y0 - 2), (55, 55, 55), 1)
    cv2.putText(right, 'Predictions', (R_LMARGIN, BAR_Y0 - 6), FONT, 0.42, LGRAY, 1)
    for rank, (idx, ten, tar, p) in enumerate(all_probs[:N_BARS]):
      y0 = BAR_Y0 + rank * BAR_ROW_H
      if y0 + BAR_ROW_H > R_H:
        break
      is_tgt = idx == target_ci
      row_clr = GOLD if is_tgt and p >= CONFIDENCE_THRESHOLD else WHITE
      bar_clr = GOLD if rank == 0 else ((0, 180, 120) if rank < 3 else (70, 110, 90))
      fill = max(1, int(R_BAR_W * p))
      ar_lbl = words_ar[idx]
      en_lbl = words_en[idx]
      cv2.rectangle(right, (R_BAR_X, y0 + 6), (R_BAR_X + R_BAR_W, y0 + BAR_ROW_H - 6), (38, 38, 38), -1)
      cv2.rectangle(right, (R_BAR_X, y0 + 6), (R_BAR_X + fill, y0 + BAR_ROW_H - 6), bar_clr, -1)
      cv2.putText(right, f'{rank + 1}.', (R_LMARGIN, y0 + 22), FONT, 0.40, row_clr, 1)
      cv2.putText(right, f'{p:.0%}', (R_BAR_X + R_BAR_W + 6, y0 + 22), FONT, 0.38, row_clr, 1)
      if ar_lbl and _has_arabic(ar_lbl):
        pil_right.append((ar_lbl, (R_LABEL_X, y0 + 2), 11, row_clr))
      cv2.putText(right, _truncate_en(en_lbl, 20), (R_LABEL_X, y0 + 30), FONT, 0.32, (175, 175, 175), 1)

    left = apply_pil_texts(left, pil_left)
    right = apply_pil_texts(right, pil_right)
    combined = np.hstack([left, right])
    cv2.line(combined, (L_W, 0), (L_W, L_H), (80, 80, 80), 2)
    cv2.imshow('Issam Arabic Words — Guided Live Test', combined)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'): break
    elif key in (ord('n'), ord('N'), 83, 100):
      tgt_en, tgt_ar, tgt_sid = load_class(target_ci + 1)
    elif key in (ord('p'), ord('P'), 81, 97):
      tgt_en, tgt_ar, tgt_sid = load_class(target_ci - 1)
    elif key in (ord('k'), ord('K')):
      ref.next_sample(); print(f'Next sample: {ref.info_str}')
    elif key in (ord('v'), ord('V')):
      if ref.videos:
        predict_sign_video(ref.videos[ref.vid_idx], expected_index=target_ci)
      else:
        vids = find_sign_videos(tgt_sid)
        if vids: predict_sign_video(vids[0], expected_index=target_ci)
    elif key in (ord('c'), ord('C')):
      buf.clear(); all_probs=[]; current_en=current_ar=''; current_conf=0.0
      print('Buffer cleared')
    elif key in (ord('g'), ord('G')):
      detect_bright = not detect_bright
      globals()['USE_DETECT_BRIGHTNESS'] = detect_bright
      print(f'Brightness: {"ON" if detect_bright else "OFF"}')
    elif key in (ord('o'), ord('O')):
      set_status(target_ci, STATUS_OK)
    elif key in (ord('r'), ord('R')):
      set_status(target_ci, STATUS_RETEST)
    elif key in (ord('u'), ord('U')):
      set_status(target_ci, STATUS_UNSET)

  summary = save_checklist(class_status)
  cap.release(); ref.close(); holistic.close(); cv2.destroyAllWindows()
  print(f'Done. Checklist saved -> {CHECKLIST_PATH.name}')
  print(f'  OK={summary["ok"]}  RETEST={summary["retest"]}  untested={summary["unset"]}')

run_guided_issam()


Class 1/100: هيكل عظمي / Skeleton  SignID 71  42 videos
N=Next  P=Prev  K=Next video  V=Video test  C=Clear  G=Brightness  Q=Quit
Class 2/100: جمجة / skull  SignID 72  42 videos
Class 3/100: عمود فقري / Backbone  SignID 73  42 videos
Brightness: OFF
Class 4/100: قفص صدري / Chest  SignID 74  42 videos
Brightness: ON
Brightness: OFF
Brightness: ON
Class 5/100: جهاز تنفسي / Respiratory device  SignID 75  42 videos
Class 6/100: قصبة هوائية / Trachea  SignID 76  42 videos
Class 7/100: رئتان / lungs  SignID 77  42 videos
Done.
